# Six passengers, before any rate

Rates come next. These six rows are the concrete cases the rates have to survive.

| Id | Name | Planet | CryoSleep | Cabin | Age | Luxury bill | Fate |
|---|---|---|---|---|---:|---:|---|
| `0001_01` | Maham Ofracculy | Europa | False | `B/0/P` | 39 | $0+0+0=0$ | stayed |
| `0002_01` | Juanna Vines | Earth | False | `F/0/S` | 24 | $109+549+44=702$ | transported |
| `0003_01` | Altark Susent | Europa | False | `A/0/S` | 58 | $43+6715+49=6807$ | stayed |
| `0003_02` | Solam Susent | Europa | False | `A/0/S` | 33 | $0+3329+193=3522$ | stayed |
| `0006_02` | Candra Jacostaffey | Earth | True | `G/0/S` | 28 | undefined | transported |
| `0013_01` | Nelly Carsoning | Earth | True | `G/3/S` | 27 | $0+0+0=0$ | unknown |

The luxury bill is `RoomService + Spa + VRDeck`. Candra's `VRDeck` cell is empty, so her luxury total is not a number, even though the bills that are present are zero. She is the first cryosleep passenger in the training file.

Altark and Solam share group `0003`, the cabin `A/0/S`, the surname Susent, and the fate "stayed." Altark is VIP. Solam is not.

Nelly is the first test passenger. Her fate is not in `test.csv`. The sample file writes `False` on her row.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "13-Spaceship Titanic" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "sample_submission.csv").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["PassengerId"], train[0]["Name"])


8693 4277 4277
0001_01 Maham Ofracculy


In [3]:
# Check the six hand readings against the stored cells.
def whole(text):
    return int(float(text))

def luxury(row):
    columns = ("RoomService", "Spa", "VRDeck")
    if any(row[column] == "" for column in columns):
        return None
    return sum(whole(row[column]) for column in columns)

by_id = {row["PassengerId"]: row for row in train}
maham = by_id["0001_01"]
juanna = by_id["0002_01"]
altark = by_id["0003_01"]
solam = by_id["0003_02"]
candra = by_id["0006_02"]
nelly = test[0]
print(maham["Name"], luxury(maham), maham["Transported"])
print(juanna["Name"], luxury(juanna), juanna["Transported"])
print(altark["Name"], luxury(altark), altark["Transported"])
print(solam["Name"], luxury(solam), solam["Transported"])
print(candra["Name"], luxury(candra), candra["VRDeck"], candra["Transported"])
print(nelly["Name"], luxury(nelly), "Transported" in nelly)
assert luxury(maham) == 0 and maham["Transported"] == "False"
assert luxury(juanna) == 109 + 549 + 44 == 702
assert juanna["Transported"] == "True"
assert luxury(altark) == 43 + 6715 + 49 == 6807
assert luxury(solam) == 0 + 3329 + 193 == 3522
assert altark["Transported"] == solam["Transported"] == "False"
assert altark["Cabin"] == solam["Cabin"] == "A/0/S"
assert candra["CryoSleep"] == "True" and candra["VRDeck"] == ""
assert luxury(candra) is None and candra["Transported"] == "True"
assert nelly["PassengerId"] == "0013_01"
assert nelly["Name"] == "Nelly Carsoning"
assert nelly["CryoSleep"] == "True" and luxury(nelly) == 0
assert submission[0]["Transported"] == "False"


Maham Ofracculy 0 False
Juanna Vines 702 True
Altark Susent 6807 False
Solam Susent 3522 False
Candra Jacostaffey None  True
Nelly Carsoning 0 False


## Pitfall

Maham spent nothing, sat on deck B, and stayed. Juanna spent 702 and was transported. Candra was in cryosleep and was transported, with one bill cell missing. A single slogan such as "a zero bill means transported" already fails the first row.

## Summary

Keep these six names. Every later rate has to be readable on Maham, Juanna, the two Susents, Candra, and Nelly.
